In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

KeyboardInterrupt: 

In [3]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
assistant.rag("How do I run Ollama locally?")

'To run Ollama locally:\n\n1. Install Ollama from: https://ollama.com/download  \n   - macOS: download the `.pkg`\n   - Windows: download the `.msi`\n   - Linux: run:\n   ```bash\n   curl -fsSL https://ollama.com/install.sh | sh\n   ```\n\n2. Open a terminal and run:\n```bash\nollama run llama3\n```\n\nThis will download the LLaMA 3 model, start it locally, and open a chat-like interface.\n\nIf you want to test that the local server is running, use:\n```bash\ncurl http://localhost:11434\n```\n\nYou should get a response like:\n```json\n{"models": [...]}\n```'

In [5]:
assistant.rag("How do I run Olama locally?")

'I don’t see any FAQ entry in the provided context about **running Olama locally**.\n\nThe closest related item is about using environment variables for API keys, but there’s no instruction here for installing or running Ollama/Olama locally.'

In [6]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [5]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [9]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

In [10]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)



[ResponseFunctionToolCall(arguments='{"query":"Can I join the course late? discovered the course can I join it enrollment late join"}', call_id='call_BUBq6sO8Z1XqRWv8NJuCbGvx', name='search', type='function_call', id='fc_0fa7155b305a5a9b006a2f098771cc81918a258b5a4b35c32d', namespace=None, status='completed')]

In [13]:
call = response.output[0]

In [14]:
import json

args = json.loads(call.arguments)

results = search(**args)
#  turn results to json
result_json = json.dumps(results, indent=2) 

In [ ]:
#  make a call to llm
#  llm decided to invoke search ('params)
# we invoke the search and hence have the results
#  send the results back to the llm (another call)
#  llm process the results
#  llm gives the answer

In [15]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output", 
    "call_id": call.call_id,
    "output": result_json,
})

In [16]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join the course.\n\nIf you want a certificate, make sure to submit your project while submissions are still open.'

In [ ]:
# Token usage and cost
usage = response.usage
usage.input_tokens, usage.output_tokens

In [ ]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

<h2> Agentic Loop</h2>

In [17]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [22]:
# execute the call
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [23]:
response.output[0]

ResponseOutputMessage(id='msg_039c7ccc4ecc826e006a2f0e947d8c819ebdf7221b7f8e8dbb', content=[ResponseOutputText(annotations=[], text='Yes, you can still join the course.\n\nIf you want to receive a certificate, make sure you submit your project while submissions are still open. Also, you can start learning and submitting homework even if you didn’t register first—registration is mainly for gauging interest.\n\nIf you want, I can also help with the course timeline, certificate rules, or how to start from here.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')

In [21]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course enroll register discover course can I join"}', call_id='call_JlHseMntQ00M9FVLgTzTRJxU', name='search', type='function_call', id='fc_039c7ccc4ecc826e006a2f0de9dfc4819eb2913d4d4d8c641e', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course enrollment eligibility join late registration FAQ"}', call_id='call_lCghXnk7Hh

In [19]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [24]:
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)

ASSISTANT:
Yes, you can still join the course.

If you want to receive a certificate, make sure you submit your project while submissions are still open. Also, you can start learning and submitting homework even if you didn’t register first—registration is mainly for gauging interest.

If you want, I can also help with the course timeline, certificate rules, or how to start from here.


In [ ]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

<h2> Full Agentic Loop </h2>

In [1]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

In [26]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [27]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally run Ollama local installation setup run locally"}
iteration #2...
function_call: search {"query":"Ollama serve localhost 11434 python client run llama3 install.sh local server"}
iteration #3...
ASSISTANT:
To run **Ollama locally**:

1. **Install Ollama**
   - macOS: download the `.pkg` from https://ollama.com/download
   - Windows: download the `.msi`
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This will download the model if needed and open a local chat session.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   If it’s working, you should get a JSON response listing models.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```
   Then:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "co

'To run **Ollama locally**:\n\n1. **Install Ollama**\n   - macOS: download the `.pkg` from https://ollama.com/download\n   - Windows: download the `.msi`\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model if needed and open a local chat session.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   If it’s working, you should get a JSON response listing models.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n   Then:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a connection issue, you may need to start the server explicitly:\n```bash\nollama serve\n```\n\nIf you want, I can also show you how to r

In [28]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join the course late can I still join discovered course FAQ enrollment late join"}
function_call: search {"query":"course enrollment late registration join after start FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still being accepted.

If you want, I can also help with:
- whether you can join in self-paced mode
- how certificates work
- what to do if you missed some homework

Is there anything else you’d like to explore?


'Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still being accepted.\n\nIf you want, I can also help with:\n- whether you can join in self-paced mode\n- how certificates work\n- what to do if you missed some homework\n\nIs there anything else you’d like to explore?'

<h2> Encouraging multiple searches </h2>

In [30]:
# simple guardrails
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course can I join discovered course enrollment add late registration FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still being accepted.

If you'd like, I can also help explain how the course participation and certificate process works. Are there other areas you want to explore?


"Yes — you can still join the course.\n\nIf you want a certificate, though, you need to submit your project while submissions are still being accepted.\n\nIf you'd like, I can also help explain how the course participation and certificate process works. Are there other areas you want to explore?"

Restricting off-topic questions

In [29]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit opening chess"}
iteration #3...
function_call: search {"query":"queen gambit chess opening"}
iteration #4...
ASSISTANT:
I couldn’t find a course FAQ entry about “queen gambit,” so it looks like this is off-topic for the course. If you meant something specific to the course, feel free to rephrase it, and I can check the FAQ again. Are there other areas you want to explore?


'I couldn’t find a course FAQ entry about “queen gambit,” so it looks like this is off-topic for the course. If you meant something specific to the course, feel free to rephrase it, and I can check the FAQ again. Are there other areas you want to explore?'

<h1> ToyAIKit </h1>

In [2]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

NameError: name 'search' is not defined

In [3]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [4]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [5]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [6]:
chat_interface = IPythonChatInterface()

In [7]:
callback = DisplayingRunnerCallback(chat_interface)

In [9]:

callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,  # for debugging purposes
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [10]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [11]:
result.cost

CostInfo(input_cost=Decimal('0.00062025'), output_cost=Decimal('0.0018855'), total_cost=Decimal('0.00250575'))

In [ ]:
result.all_messages

In [12]:
result = runner.loop(
    prompt="How do I run a different model",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


In [13]:
runner.run();